# Pix2Pix / Attention U-Net — Traduction d'images médicales
## Données MMIT-DDPM BraTS

Ce notebook entraîne un modèle de traduction conditionnelle d'images sur les tranches 2D
pré-traitées du pipeline MMIT-DDPM. Trois architectures disponibles :

| `model_type`  | Architecture                                         |
|---------------|------------------------------------------------------|
| `unet`        | U-Net vanille (régression, pas de GAN)               |
| `pix2pix`     | U-Net + PatchGAN 70×70 (Isola et al. 2017)           |
| `attn_cgan`   | Attention U-Net + PatchGAN multi-échelle *(défaut)*  |

**Paires disponibles** : `ft1ce` · `ft2` · `t1cet2` · `t1cetf` · `t2f` · `t2t1ce`  
**Format** : `.nii.gz` 2D, normalisé [-1, 1], shape `(H, W, 2)` — canal 0 = source, canal 1 = cible.

## 0 — Cloner le dépôt & installer les dépendances

Exécuter cette cellule une seule fois (Colab / Kaggle / toute machine distante).  
Si vous exécutez localement et que le dépôt est déjà présent, la cellule détecte le dossier et passe.

In [ ]:
import os, sys

REPO_URL  = 'https://github.com/hashirama21/mmit-ddpm.git'
REPO_NAME = 'mmit-ddpm'

if not os.path.isdir(REPO_NAME):
    os.system(f'git clone {REPO_URL}')

# Move into the repo so all relative paths (./data/, ./outputs/) work
if os.path.basename(os.getcwd()) != REPO_NAME:
    os.chdir(REPO_NAME)

print('Working directory:', os.getcwd())

os.system('pip install -q -r requirements.txt')
os.system('pip install -q nibabel scikit-image pytorch_msssim torchvision tqdm')
print('Dependencies installed.')

## 1 — Imports & device

In [ ]:
from __future__ import annotations
import os, random, warnings
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Literal, Optional, Tuple

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio as skimage_psnr
from skimage.metrics import structural_similarity  as skimage_ssim
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

warnings.filterwarnings('ignore')

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})

print(f'PyTorch {torch.__version__}  |  Device: {DEVICE}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  —  {p.total_memory/1e9:.1f} GB VRAM')

## 2 — Configuration

**Seule section à modifier avant de lancer l'entraînement.**  
Régler `dataset_fraction` pour contrôler la taille du sous-ensemble utilisé :
- `0.05` → test rapide du pipeline (quelques patients, ~2 min)
- `0.20` → entraînement partiel pour valider les métriques
- `1.0`  → entraînement complet

In [ ]:
# ══════════════════════════════════════════════════════════
#  PARAMÈTRES  — modifier ici avant de lancer
# ══════════════════════════════════════════════════════════

TRANSLATION_PAIR  = 'ft1ce'      # Flair -> T1CE  (changer pour toute autre paire)
MODEL_TYPE        = 'attn_cgan'  # 'unet' | 'pix2pix' | 'attn_cgan'

DATASET_FRACTION  = 0.2          # fraction du dataset d'entraînement (0.0–1.0)
MAX_EPOCHS        = 100
BATCH_SIZE        = 8
NUM_WORKERS       = 2            # 0 si problème de mémoire partagée sur Colab

# ══════════════════════════════════════════════════════════

PAIR_LABELS = {
    'ft1ce': ('Flair', 'T1CE'), 'ft2':    ('Flair', 'T2'),
    't1cet2': ('T1CE', 'T2'),   't1cetf': ('T1CE', 'Flair'),
    't2f':   ('T2',   'Flair'), 't2t1ce': ('T2',   'T1CE'),
}


@dataclass
class Config:
    # Chemins
    train_dir    : Path = Path('./data/training')
    test_dir     : Path = Path('./data/testing')
    project_root : Path = Path('.')

    # Tâche
    translation_pair : str = TRANSLATION_PAIR
    in_c             : int = 1   # canaux source (1 par paire)

    # Fraction du dataset  ← paramètre clé
    dataset_fraction : float = DATASET_FRACTION
    train_frac       : float = 0.80   # train / val depuis ./data/training
    split_seed       : int   = 42

    # Architecture
    model_type : Literal['unet', 'pix2pix', 'attn_cgan'] = MODEL_TYPE
    feats      : Tuple[int, ...] = (64, 128, 256, 512)
    dropout    : float = 0.3
    ndf        : int   = 64

    # Entraînement
    max_epochs        : int   = MAX_EPOCHS
    batch_size        : int   = BATCH_SIZE
    lr_g              : float = 2e-4
    lr_d              : float = 1e-4
    n_gen_steps       : int   = 2
    grad_clip         : float = 1.0
    val_every         : int   = 5
    patience          : int   = 15
    num_workers       : int   = NUM_WORKERS
    tumour_oversample : float = 3.0

    # Poids de la loss
    w_adv         : float = 1.0
    w_l1          : float = 100.0
    w_perc        : float = 10.0
    w_freq        : float = 5.0
    w_ssim        : float = 10.0
    lambda_tumour : float = 5.0

    # EMA & augmentation
    ema_decay       : float = 0.999
    aug_flip_p      : float = 0.5
    aug_vflip_p     : float = 0.5
    aug_rot_p       : float = 0.5
    aug_max_angle   : float = 15.0
    aug_intensity_p : float = 0.5

    # Métriques (données normalisées [-1,1] => plage fixe = 2.0)
    metric_data_range : float = 2.0

    # Sortie
    out_name : str = f'run_{TRANSLATION_PAIR}_{MODEL_TYPE}'

    @property
    def splits_csv(self) -> Path: return self.project_root / 'splits.csv'
    @property
    def out_dir(self)    -> Path: return self.project_root / 'outputs' / self.out_name


CFG = Config()

src, tgt = PAIR_LABELS.get(CFG.translation_pair, ('Source', 'Cible'))
pct       = f'{CFG.dataset_fraction:.0%}'

print(f'Paire       : {CFG.translation_pair}  ({src} → {tgt})')
print(f'Modèle      : {CFG.model_type}')
print(f'Dataset     : {pct} des patients d\'entraînement')
print(f'Epochs      : {CFG.max_epochs}  |  Batch : {CFG.batch_size}')
print(f'Sortie      : {CFG.out_dir}')

## 3 — Pipeline de données

Lecture directe des fichiers `.nii.gz` — aucun pré-traitement supplémentaire nécessaire.  
Le pipeline MMIT-DDPM a déjà normalisé toutes les tranches dans [-1, 1].

In [ ]:
def parse_mmit_filename(fname: str) -> Optional[Dict]:
    """Parse brats_{split}_{pid}_{pair}_{slice}_w.nii.gz"""
    stem  = fname.replace('.nii.gz', '').replace('_w', '')
    parts = stem.split('_')
    if len(parts) < 5 or parts[0] != 'brats':
        return None
    for i, p in enumerate(parts):
        if p in PAIR_LABELS:
            return {'patient_id': '_'.join(parts[2:i]), 'pair': p, 'slice': parts[i+1]}
    return None


class PairedAugmenter:
    """Augmentation géométrique + intensité compatible avec in_c quelconque."""

    def __init__(self, cfg: Config):
        self.fp = cfg.aug_flip_p; self.vp = cfg.aug_vflip_p
        self.rp = cfg.aug_rot_p;  self.ma = cfg.aug_max_angle
        self.ip = cfg.aug_intensity_p

    def _jitter(self, x):
        x  = x + random.uniform(-0.15, 0.15)
        mu = x.mean()
        x  = (x - mu) * random.uniform(0.8, 1.2) + mu
        g  = random.uniform(0.8, 1.2)
        return (((x.clamp(-1,1)+1)/2).clamp(0,1).pow(g)*2-1).clamp(-1,1)

    def __call__(self, x, y, seg):
        import torchvision.transforms.functional as TF
        ic  = x.shape[0]
        cat = torch.cat([x, y, seg.float()], 0)
        if random.random() < self.fp:  cat = torch.flip(cat, [2])
        if random.random() < self.vp:  cat = torch.flip(cat, [1])
        if random.random() < self.rp:
            cat = TF.rotate(cat, random.uniform(-self.ma, self.ma),
                            interpolation=TF.InterpolationMode.BILINEAR)
        x_o, y_o, s_o = cat[:ic], cat[ic:ic+1], (cat[ic+1:ic+2]>0.5).float()
        if random.random() < self.ip: x_o = self._jitter(x_o)
        return x_o, y_o, s_o


class MMITBraTS2DDataset(Dataset):
    """
    Lit les tranches MMIT-DDPM directement depuis les fichiers .nii.gz.
    Retourne (source, cible, masque_cerveau, meta).
    """

    def __init__(self, data_dir, patient_ids, pair, augment=False, cfg=None):
        self.aug  = PairedAugmenter(cfg) if (augment and cfg) else None
        pid_set   = set(patient_ids)
        self.files, self.meta = [], []
        for f in sorted(Path(data_dir).glob(f'*_{pair}_*_w.nii.gz')):
            info = parse_mmit_filename(f.name)
            if info and info['patient_id'] in pid_set:
                self.files.append(f)
                self.meta.append({'pid': info['patient_id'], 'z': int(info['slice'])})
        # Toutes les tranches BraTS proviennent de patients tumoraux
        self.has_tumour_flags = [True] * len(self.files)
        self.tumour_fracs     = [1.0]  * len(self.files)

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        vol = nib.load(str(self.files[idx])).get_fdata().astype(np.float32)
        src = torch.from_numpy(vol[:,:,0]).unsqueeze(0)  # (1, H, W)
        tgt = torch.from_numpy(vol[:,:,1]).unsqueeze(0)  # (1, H, W)
        seg = (tgt.abs() > 0.05).float()                 # masque cerveau
        if self.aug: src, tgt, seg = self.aug(src, tgt, seg)
        return src, tgt, seg, {'pid': self.meta[idx]['pid'],
                               'z':   self.meta[idx]['z'],
                               'has_tumour': True}


class SplitManager:
    """
    Découpe patient-level 80/20 (train/val) depuis ./data/training/
    + test fixe depuis ./data/testing/.
    dataset_fraction appliqué de manière reproductible.
    """

    def __init__(self, cfg: Config): self.cfg = cfg

    def _patients(self, d: Path) -> List[str]:
        pids = set()
        for f in Path(d).glob(f'*_{self.cfg.translation_pair}_*_w.nii.gz'):
            info = parse_mmit_filename(f.name)
            if info: pids.add(info['patient_id'])
        return sorted(pids)

    def make(self, force=False):
        if self.cfg.splits_csv.exists() and not force:
            return
        tr_pids = self._patients(self.cfg.train_dir)
        te_pids = self._patients(self.cfg.test_dir)
        if not tr_pids:
            raise RuntimeError(f"Aucun fichier '{self.cfg.translation_pair}' dans {self.cfg.train_dir}")
        rng = np.random.default_rng(self.cfg.split_seed)
        pids = list(rng.permutation(tr_pids))
        t    = max(1, int(len(pids) * self.cfg.train_frac))
        rows = (
            [{'patient_id': p, 'split': 'train', 'data_dir': str(self.cfg.train_dir)} for p in pids[:t]]
          + [{'patient_id': p, 'split': 'val',   'data_dir': str(self.cfg.train_dir)} for p in pids[t:]]
          + [{'patient_id': p, 'split': 'test',  'data_dir': str(self.cfg.test_dir)}  for p in te_pids]
        )
        pd.DataFrame(rows).to_csv(self.cfg.splits_csv, index=False)
        print(f'splits.csv créé — train:{t}  val:{len(pids)-t}  test:{len(te_pids)}')

    def load(self) -> Dict:
        if not self.cfg.splits_csv.exists(): self.make()
        df = pd.read_csv(self.cfg.splits_csv)
        sp = defaultdict(lambda: {'pids': [], 'data_dir': None})
        for _, r in df.iterrows():
            sp[r['split']]['pids'].append(r['patient_id'])
            sp[r['split']]['data_dir'] = Path(r['data_dir'])
        frac = self.cfg.dataset_fraction
        if frac < 1.0:
            rng = np.random.default_rng(self.cfg.split_seed + 1)
            for k in sp:
                p = list(rng.permutation(sp[k]['pids']))
                sp[k]['pids'] = p[:max(1, int(len(p)*frac))]
        return dict(sp)


class DataLoaderFactory:
    def __init__(self, cfg: Config): self.cfg = cfg

    def build(self, splits: Dict) -> Dict:
        loaders = {}
        for name, info in splits.items():
            is_tr = name == 'train'
            ds    = MMITBraTS2DDataset(info['data_dir'], info['pids'],
                                       self.cfg.translation_pair, augment=is_tr, cfg=self.cfg)
            if not ds: print(f'  [{name}] AVERTISSEMENT: 0 tranches'); continue
            kw = dict(batch_size=self.cfg.batch_size, num_workers=self.cfg.num_workers,
                      pin_memory=DEVICE.type=='cuda',
                      persistent_workers=self.cfg.num_workers>0,
                      prefetch_factor=2 if self.cfg.num_workers>0 else None)
            if is_tr:
                w  = [1+(self.cfg.tumour_oversample-1)*min(tf*20,1) for tf in ds.tumour_fracs]
                loaders[name] = DataLoader(ds, sampler=WeightedRandomSampler(w, len(w), True),
                                           drop_last=True, **kw)
            else:
                loaders[name] = DataLoader(ds, shuffle=False, **kw)
            print(f'  [{name:5s}]  {len(info["pids"]):3d} patients  |  {len(ds):,} tranches')
        return loaders


print('Pipeline données prêt.')

## 4 — Exploration des données

Vérifie que les fichiers sont présents et visualise quelques paires source/cible.

In [ ]:
def scan(data_dir, pair):
    files = sorted(Path(data_dir).glob(f'*_{pair}_*_w.nii.gz'))
    pids  = {parse_mmit_filename(f.name)['patient_id']
             for f in files if parse_mmit_filename(f.name)}
    label = f'{pair}' + (f' ({PAIR_LABELS[pair][0]}→{PAIR_LABELS[pair][1]})' if pair in PAIR_LABELS else '')
    print(f'{Path(data_dir).name:12s}  {len(files):4d} tranches  {len(pids):3d} patients  [{label}]')
    if files:
        vol = nib.load(str(files[0])).get_fdata()
        print(f'  shape={vol.shape}  range=[{vol.min():.3f}, {vol.max():.3f}]')
    return files, pids

print('─' * 60)
tr_files, tr_pids = scan(CFG.train_dir, CFG.translation_pair)
te_files, te_pids = scan(CFG.test_dir,  CFG.translation_pair)
print('─' * 60)

# Visualisation de 4 échantillons aléatoires
if tr_files:
    sample_files = [tr_files[i] for i in np.linspace(0, len(tr_files)-1, min(4, len(tr_files)), dtype=int)]
    src_name, tgt_name = PAIR_LABELS.get(CFG.translation_pair, ('Source', 'Cible'))
    fig, axes = plt.subplots(2, len(sample_files), figsize=(5*len(sample_files), 10))
    if len(sample_files) == 1: axes = axes.reshape(2, 1)
    for col, f in enumerate(sample_files):
        vol = nib.load(str(f)).get_fdata().astype(np.float32)
        axes[0, col].imshow(vol[:,:,0], cmap='gray', vmin=-1, vmax=1)
        axes[0, col].set_title(f'Source: {src_name}', fontsize=9); axes[0, col].axis('off')
        axes[1, col].imshow(vol[:,:,1], cmap='gray', vmin=-1, vmax=1)
        axes[1, col].set_title(f'Cible: {tgt_name}', fontsize=9);  axes[1, col].axis('off')
    plt.suptitle(f'Échantillons MMIT-DDPM  —  {CFG.translation_pair}', fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

## 5 — Architectures

In [ ]:
# ── Blocs de base ─────────────────────────────────────────────────────────────

class DoubleConv(nn.Module):
    def __init__(self, ic, oc, norm='instance', drop=0.0):
        super().__init__()
        N = lambda c: nn.InstanceNorm2d(c, affine=True) if norm=='instance' else nn.GroupNorm(min(8,c),c)
        L = [nn.Conv2d(ic,oc,3,padding=1,bias=False), N(oc), nn.LeakyReLU(0.2,True),
             nn.Conv2d(oc,oc,3,padding=1,bias=False), N(oc), nn.LeakyReLU(0.2,True)]
        if drop: L.append(nn.Dropout2d(drop))
        self.b = nn.Sequential(*L)
    def forward(self, x): return self.b(x)


class AttentionGate(nn.Module):
    """Oktay et al. 2018 — alpha = sigmoid(psi(ReLU(Wg(g)+Wx(x))))"""
    def __init__(self, Fg, Fl, Fi):
        super().__init__()
        self.Wg  = nn.Sequential(nn.Conv2d(Fg,Fi,1,bias=True), nn.BatchNorm2d(Fi))
        self.Wx  = nn.Sequential(nn.Conv2d(Fl,Fi,1,bias=True), nn.BatchNorm2d(Fi))
        self.psi = nn.Sequential(nn.Conv2d(Fi,1, 1,bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(True)
    def forward(self, g, x):
        g1 = self.Wg(g)
        if g1.shape[2:] != x.shape[2:]:
            g1 = F.interpolate(g1, x.shape[2:], mode='bilinear', align_corners=False)
        return x * self.psi(self.relu(g1 + self.Wx(x)))


class DiscBlock(nn.Module):
    def __init__(self, ic, oc, stride=2, norm=True):
        super().__init__()
        L = [nn.Conv2d(ic,oc,4,stride=stride,padding=1,bias=not norm)]
        if norm: L.append(nn.InstanceNorm2d(oc, affine=True))
        self.b = nn.Sequential(*L, nn.LeakyReLU(0.2,True))
    def forward(self, x): return self.b(x)


# ── Générateurs ───────────────────────────────────────────────────────────────

class BaseUNet(nn.Module):
    def __init__(self, ic=1, oc=1, feats=(64,128,256,512), tanh=True):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.enc  = nn.ModuleList(); ch=ic
        for f in feats: self.enc.append(DoubleConv(ch,f)); ch=f
        self.bn   = DoubleConv(ch, ch*2); ch=ch*2
        self.up   = nn.ModuleList(); self.dec = nn.ModuleList()
        for f in reversed(feats):
            self.up.append(nn.Sequential(nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                                         nn.Conv2d(ch,f,1,bias=False)))
            self.dec.append(DoubleConv(f*2,f)); ch=f
        hd = [nn.Conv2d(ch,oc,1)]
        if tanh: hd.append(nn.Tanh())
        self.head = nn.Sequential(*hd)

    def forward(self, x):
        sk=[]
        for e in self.enc: x=e(x); sk.append(x); x=self.pool(x)
        x = self.bn(x)
        for u,d,s in zip(self.up, self.dec, reversed(sk)):
            x=u(x)
            if x.shape[2:]!=s.shape[2:]: x=F.interpolate(x,s.shape[2:],mode='bilinear',align_corners=False)
            x=d(torch.cat([s,x],1))
        return self.head(x)

    @property
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)


class AttentionUNet(nn.Module):
    def __init__(self, ic=1, oc=1, feats=(64,128,256,512), drop=0.3):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.enc  = nn.ModuleList(); ch=ic
        for f in feats: self.enc.append(DoubleConv(ch,f)); ch=f
        self.bn = DoubleConv(ch,ch*2); ch=ch*2
        self.ag = nn.ModuleList(); self.up = nn.ModuleList(); self.dec = nn.ModuleList()
        gc = ch
        for i,f in enumerate(reversed(feats)):
            self.ag.append(AttentionGate(gc,f,f//2))
            self.up.append(nn.Sequential(nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
                                         nn.Conv2d(gc,f,1,bias=False)))
            self.dec.append(DoubleConv(f*2,f, drop=drop if i<2 else 0.0)); gc=f
        self.head = nn.Sequential(nn.Conv2d(gc,oc,1), nn.Tanh())

    def forward(self, x):
        sk=[]
        for e in self.enc: x=e(x); sk.append(x); x=self.pool(x)
        x = self.bn(x)
        for u,ag,d,s in zip(self.up, self.ag, self.dec, reversed(sk)):
            sa=ag(x,s); x=u(x)
            if x.shape[2:]!=sa.shape[2:]: x=F.interpolate(x,sa.shape[2:],mode='bilinear',align_corners=False)
            x=d(torch.cat([sa,x],1))
        return self.head(x)

    @property
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ── Discriminateurs ───────────────────────────────────────────────────────────

class PatchGANDiscriminator(nn.Module):
    """PatchGAN conditionnel 70×70 (Isola et al. 2017)."""
    def __init__(self, ic=1, ndf=64):
        super().__init__()
        self.m = nn.Sequential(
            DiscBlock(ic+1,ndf,2,False), DiscBlock(ndf,ndf*2,2),
            DiscBlock(ndf*2,ndf*4,2),   DiscBlock(ndf*4,ndf*8,1),
            nn.Conv2d(ndf*8,1,4,1,1))
    def forward(self,x,y): return self.m(torch.cat([x,y],1))
    @property
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)


class MultiScaleDiscriminator(nn.Module):
    """Deux PatchGAN à résolution originale et ×2 sous-échantillonnée."""
    def __init__(self, ic=1, ndf=64):
        super().__init__()
        self.D1=PatchGANDiscriminator(ic,ndf); self.D2=PatchGANDiscriminator(ic,ndf)
        self.dn=nn.AvgPool2d(2,2)
    def forward(self,x,y): return [self.D1(x,y), self.D2(self.dn(x),self.dn(y))]
    @property
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)


def build_generator(cfg):
    if cfg.model_type=='unet': return BaseUNet(ic=cfg.in_c, oc=1, feats=cfg.feats)
    return AttentionUNet(ic=cfg.in_c, oc=1, feats=cfg.feats, drop=cfg.dropout)

def build_discriminator(cfg):
    if cfg.model_type=='unet':    return None
    if cfg.model_type=='pix2pix': return PatchGANDiscriminator(ic=cfg.in_c, ndf=cfg.ndf)
    return MultiScaleDiscriminator(ic=cfg.in_c, ndf=cfg.ndf)


# Vérification rapide
g = build_generator(CFG)
with torch.no_grad():
    o = g(torch.randn(1, CFG.in_c, 240, 240))
print(f'Générateur ({CFG.model_type}) : {g.n_params:,} params  →  out {tuple(o.shape)}')
del g, o

## 6 — Fonctions de perte

In [ ]:
def tumour_weighted_l1(pred, tgt, seg, lt=5.0):
    if seg is None or lt<=1: return F.l1_loss(pred, tgt)
    return ((1 + (lt-1)*(seg>0).float()) * (pred-tgt).abs()).mean()

def freq_domain_loss(pred, gt):
    return F.l1_loss(torch.abs(torch.fft.rfft2(pred.float(), norm='ortho')),
                     torch.abs(torch.fft.rfft2(gt.float(),   norm='ortho')))

def disc_hinge(rs, fs):
    L = torch.tensor(0., device=rs[0].device)
    for r,f in zip(rs,fs): L = L + F.relu(1.-r).mean() + F.relu(1.+f).mean()
    return L/len(rs)

def gen_adv(fs): return sum(-f.mean() for f in fs)/len(fs)


class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        try:
            import torchvision.models as tvm
            vgg = tvm.vgg16(weights=tvm.VGG16_Weights.DEFAULT).features.eval()
            self.sl = nn.ModuleList([
                nn.Sequential(*list(vgg.children())[:4]),
                nn.Sequential(*list(vgg.children())[4:9])])
            for p in self.parameters(): p.requires_grad_(False)
            self.register_buffer('m', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
            self.register_buffer('s', torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
            self._ok = True
        except Exception as e: print(f'[VGG] indisponible: {e}'); self._ok=False

    def _prep(self, x):
        x = (x+1)/2
        if x.shape[1]==1: x=x.repeat(1,3,1,1)
        return (x-self.m)/self.s

    def forward(self, pred, gt):
        if not self._ok: return torch.tensor(0., device=pred.device)
        p,g = self._prep(pred), self._prep(gt)
        L = torch.tensor(0., device=pred.device)
        for sl in self.sl: p=sl(p); g=sl(g).detach(); L=L+F.l1_loss(p,g)
        return L


try:
    from pytorch_msssim import ssim as _pssim; _SSIM='pytorch_msssim'
except ImportError: _SSIM='fallback'

def _gwin(sz, sig, dev, dt):
    c = torch.arange(sz,dtype=dt,device=dev)-sz//2
    g = torch.exp(-(c**2)/(2*sig**2)); g/=g.sum()
    return (g.unsqueeze(0)*g.unsqueeze(1)).unsqueeze(0).unsqueeze(0)

def _ssim_fb(p, g, dr=2.0, ws=11):
    C1,C2=(0.01*dr)**2,(0.03*dr)**2
    w=_gwin(ws,1.5,p.device,p.dtype).expand(p.shape[1],1,ws,ws); pad=ws//2
    f=lambda x: F.conv2d(x,w,padding=pad,groups=x.shape[1])
    mp,mg=f(p),f(g)
    return (((2*mp*mg+C1)*(2*(f(p*g)-mp*mg)+C2))/
            ((mp**2+mg**2+C1)*(f(p*p)-mp**2+f(g*g)-mg**2+C2))).mean()

def ssim_loss(p, g, dr=2.0):
    p,g=p.float(),g.float()
    return 1-(_pssim(p,g,data_range=dr,size_average=True) if _SSIM=='pytorch_msssim' else _ssim_fb(p,g,dr))


class CompositeLoss(nn.Module):
    """
    L = w_adv·L_cGAN + w_l1·L1 + w_perc·L_VGG + w_freq·L_FFT + w_ssim·(1−SSIM)
    w_adv = 0 automatiquement pour model_type='unet'.
    """
    def __init__(self, cfg):
        super().__init__()
        self.wa = cfg.w_adv if cfg.model_type!='unet' else 0.
        self.wl,self.wp,self.wf,self.ws = cfg.w_l1, cfg.w_perc, cfg.w_freq, cfg.w_ssim
        self.lt, self.dr = cfg.lambda_tumour, cfg.metric_data_range
        self.vgg = PerceptualLoss()

    def forward(self, fs, pred, gt, seg=None):
        L = {
            'adv':  (gen_adv(fs)*self.wa if self.wa>0 and fs else torch.tensor(0.,device=pred.device)),
            'l1':   tumour_weighted_l1(pred,gt,seg,self.lt)*self.wl,
            'perc': self.vgg(pred,gt)*self.wp,
            'freq': freq_domain_loss(pred,gt)*self.wf,
            'ssim': ssim_loss(pred,gt,self.dr)*self.ws,
        }
        tot = sum(L.values())
        return tot, {k: float(v.item()) for k,v in L.items()}


print(f'Fonctions de perte prêtes. Backend SSIM: {_SSIM}')

## 7 — Boucle d'entraînement

In [ ]:
class MetricsTracker:
    def __init__(self, dr=2.0): self.dr=dr; self.rec=[]
    def reset(self): self.rec.clear()
    def update(self, p, g, ht):
        m = g!=0
        if not m.any(): return
        self.rec.append({'ssim': float(skimage_ssim(p,g,data_range=self.dr)),
                         'psnr': float(skimage_psnr(g,p,data_range=self.dr)),
                         'mae':  float(np.abs(p[m]-g[m]).mean()), 'ht': ht})
    def _agg(self, sub, k):
        v=[r[k] for r in sub if np.isfinite(r[k])]
        return {'mean':float(np.mean(v)),'std':float(np.std(v)),'n':len(v)} if v \
               else {'mean':float('nan'),'std':float('nan'),'n':0}
    def summary(self):
        return {g: {k: self._agg([r for r in self.rec if (g=='all' or bool(r['ht'])==(g=='tumour'))], k)
                    for k in ('ssim','psnr','mae')}
                for g in ('all','tumour','healthy')}
    def ssim_mean(self):
        v=[r['ssim'] for r in self.rec if np.isfinite(r['ssim'])]
        return float(np.mean(v)) if v else 0.


class EarlyStopping:
    def __init__(self, p=15, d=5e-4): self.p=p; self.d=d; self.best=-np.inf; self.cnt=0; self.stop=False
    def __call__(self, s):
        if s>self.best+self.d: self.best=s; self.cnt=0
        else:
            self.cnt+=1
            if self.cnt>=self.p: self.stop=True
        return self.stop


class Trainer:
    """
    Boucle complète pour unet / pix2pix / attn_cgan.
    AMP · gradient clipping · EMA · décroissance LR linéaire ·
    sauvegarde meilleur checkpoint · log CSV.
    """

    def __init__(self, cfg):
        self.cfg=cfg; cfg.out_dir.mkdir(parents=True,exist_ok=True)
        self.ckpt=cfg.out_dir/'best.pth'; self.log=cfg.out_dir/'train_log.csv'

    def _init(self):
        gen=build_generator(self.cfg).to(DEVICE)
        disc=build_discriminator(self.cfg)
        if disc: disc=disc.to(DEVICE)
        def _wi(m):
            if isinstance(m,(nn.Conv2d,nn.ConvTranspose2d)): nn.init.normal_(m.weight,0,.02)
            elif isinstance(m,(nn.BatchNorm2d,nn.InstanceNorm2d)):
                if hasattr(m,'weight') and m.weight is not None: nn.init.normal_(m.weight,1,.02)
                if hasattr(m,'bias')   and m.bias   is not None: nn.init.constant_(m.bias,0)
        gen.apply(_wi)
        if disc: disc.apply(_wi)
        print(f'Générateur    : {gen.n_params:,} params')
        if disc: print(f'Discriminateur: {disc.n_params:,} params')
        try:
            gen=torch.compile(gen,mode='reduce-overhead')
            if disc: disc=torch.compile(disc,mode='default')
            print('[torch.compile] actif')
        except: pass
        return gen, disc

    def _opts(self, gen, disc):
        og=torch.optim.Adam(gen.parameters(), lr=self.cfg.lr_g, betas=(.5,.999))
        od=torch.optim.Adam(disc.parameters(),lr=self.cfg.lr_d, betas=(.5,.999)) if disc else None
        hs=self.cfg.max_epochs//2
        lm=lambda e: 1. if e<hs else max(0.,1.-(e-hs)/hs)
        return og,od,torch.optim.lr_scheduler.LambdaLR(og,lm),\
               (torch.optim.lr_scheduler.LambdaLR(od,lm) if od else None)

    def _save(self, ep, gen, disc, og, od, met, sg, sd, ema):
        sd_=lambda m:(m._orig_mod if hasattr(m,'_orig_mod') else m).state_dict()
        p={'epoch':ep,'gen':sd_(gen),'metrics':met}
        if disc: p['disc']=sd_(disc)
        if og:   p['opt_g']=og.state_dict()
        if od:   p['opt_d']=od.state_dict()
        if sg:   p['sg']=sg.state_dict()
        if sd:   p['sd']=sd.state_dict()
        if ema:  p['gen_ema']=sd_(ema)
        torch.save(p,self.ckpt)

    def load(self, gen, disc, og=None, od=None, sg=None, sd=None, ema=None):
        if not self.ckpt.exists(): return 0
        c=torch.load(self.ckpt,map_location=DEVICE,weights_only=False)
        t=lambda m:(m._orig_mod if hasattr(m,'_orig_mod') else m)
        t(gen).load_state_dict(c['gen'])
        if disc and 'disc' in c: t(disc).load_state_dict(c['disc'])
        if og and 'opt_g' in c:  og.load_state_dict(c['opt_g'])
        if od and 'opt_d' in c:  od.load_state_dict(c['opt_d'])
        if sg and 'sg'    in c:  sg.load_state_dict(c['sg'])
        if sd and 'sd'    in c:  sd.load_state_dict(c['sd'])
        if ema and 'gen_ema' in c: t(ema).load_state_dict(c['gen_ema'])
        ep=int(c.get('epoch',0))
        print(f'[Checkpoint] epoch={ep}  SSIM={c["metrics"].get("ssim_all",0):.4f}')
        return ep

    @torch.no_grad()
    def _val(self, gen, dl):
        gen.eval(); tr=MetricsTracker(self.cfg.metric_data_range)
        for x,y,seg,meta in dl:
            x=x.to(DEVICE)
            with autocast(device_type=AMP_DEVICE): pred=gen(x)
            pn,yn=pred.cpu().float().numpy(),y.float().numpy()
            for b in range(pn.shape[0]):
                ht=bool(meta['has_tumour'][b]) if isinstance(meta['has_tumour'],torch.Tensor) else meta['has_tumour'][b]
                tr.update(pn[b,0],yn[b,0],ht)
        return tr.summary()

    def fit(self, loaders, resume=False):
        gen,disc=self._init()
        og,od,sch_g,sch_d=self._opts(gen,disc)
        crit=CompositeLoss(self.cfg).to(DEVICE)
        sg,sd=GradScaler(device=AMP_DEVICE),GradScaler(device=AMP_DEVICE)
        es=EarlyStopping(self.cfg.patience)
        best_ssim,best_m,rows=-np.inf,{},[]; ep0=1

        # Copie EMA du générateur
        ema=build_generator(self.cfg).to(DEVICE)
        tg=lambda m:(m._orig_mod if hasattr(m,'_orig_mod') else m)
        tg(ema).load_state_dict(tg(gen).state_dict())
        for p in ema.parameters(): p.requires_grad_(False)
        dc=self.cfg.ema_decay

        @torch.no_grad()
        def _ema():
            src,dst=tg(gen).state_dict(),tg(ema).state_dict()
            for k in dst:
                if dst[k].dtype.is_floating_point: dst[k].mul_(dc).add_(src[k],alpha=1-dc)
                else: dst[k].copy_(src[k])

        if resume: ep0=self.load(gen,disc,og,od,sg,sd,ema)+1

        for ep in range(ep0, self.cfg.max_epochs+1):
            gen.train()
            if disc: disc.train()
            sg_,sd_,n=0.,0.,0
            bar=tqdm(loaders['train'],desc=f'Ep {ep:3d}/{self.cfg.max_epochs}',ncols=100,leave=False)

            for x,y,seg,_ in bar:
                x,y,seg=x.to(DEVICE),y.to(DEVICE),seg.to(DEVICE)

                if disc and od:
                    with autocast(device_type=AMP_DEVICE):
                        yf=gen(x)
                        rs=disc(x,y);           rs=rs if isinstance(rs,list) else [rs]
                        fs=disc(x,yf.detach()); fs=fs if isinstance(fs,list) else [fs]
                        ld=disc_hinge(rs,fs)
                    od.zero_grad(set_to_none=True); sd.scale(ld).backward()
                    sd.unscale_(od); nn.utils.clip_grad_norm_(disc.parameters(),self.cfg.grad_clip)
                    sd.step(od); sd.update(); sd_+=ld.item()

                for _ in range(self.cfg.n_gen_steps if disc else 1):
                    with autocast(device_type=AMP_DEVICE):
                        yf=gen(x)
                        fk=disc(x,yf) if disc else None
                        fl=fk if isinstance(fk,list) else ([fk] if fk is not None else None)
                        lg,_=crit(fl,yf,y,seg)
                    og.zero_grad(set_to_none=True); sg.scale(lg).backward()
                    sg.unscale_(og); nn.utils.clip_grad_norm_(gen.parameters(),self.cfg.grad_clip)
                    sg.step(og); sg.update(); _ema()

                sg_+=lg.item(); n+=1
                bar.set_postfix({'G':f'{lg.item():.3f}','D':f'{sd_/n:.3f}' if disc else '-',
                                 'lr':f'{og.param_groups[0]["lr"]:.1e}'})

            sch_g.step()
            if sch_d: sch_d.step()
            ag,ad=sg_/max(n,1),sd_/max(n,1)

            if ep%self.cfg.val_every==0:
                stats=self._val(ema,loaders['val'])
                sa=stats['all']['ssim']['mean']; sp=stats['all']['psnr']['mean']
                print(f'[{ep:3d}] G={ag:.3f} D={ad:.3f}  SSIM={sa:.4f}  PSNR={sp:.2f}dB')
                if sa>best_ssim+5e-4:
                    best_ssim=sa
                    best_m={'epoch':ep,'ssim_all':sa,'psnr_all':sp}
                    self._save(ep,gen,disc,og,od,best_m,sg,sd,ema)
                    print(f'     ✓ nouveau meilleur SSIM={best_ssim:.4f}  (checkpoint sauvegardé)')
                rows.append({'epoch':ep,'loss_g':ag,'loss_d':ad,'ssim':sa,'psnr':sp})
                pd.DataFrame(rows).to_csv(self.log,index=False)
                if es(sa): print(f'[EarlyStopping] Pas d\'amélioration depuis {self.cfg.patience} validations.'); break
            else:
                print(f'[{ep:3d}] G={ag:.3f}  D={ad:.3f}')

        print(f'\nMeilleur SSIM = {best_ssim:.4f}  (epoch {best_m.get("epoch","-")})')
        return best_m


print('Trainer prêt.')

## 8 — Préparer les données & lancer l'entraînement

Toute la logique est dans cette cellule :  
1. Génération des splits patients (80 % train / 20 % val + test séparé)  
2. Construction des DataLoaders avec `dataset_fraction`  
3. Lancement du `Trainer`  

Aucune autre cellule n'est nécessaire pour entraîner le modèle.

In [ ]:
# ── 1. Splits ────────────────────────────────────────────────────────────────
split_mgr = SplitManager(CFG)
split_mgr.make()          # idempotent — ignoré si splits.csv existe déjà
splits    = split_mgr.load()

print(f'\nFraction utilisée : {CFG.dataset_fraction:.0%}')
for k, v in splits.items():
    print(f'  {k:5s} : {len(v["pids"]):3d} patients')

# ── 2. DataLoaders ───────────────────────────────────────────────────────────
print('\nConstruction des DataLoaders...')
loaders = DataLoaderFactory(CFG).build(splits)

# ── 3. Entraînement ──────────────────────────────────────────────────────────
print(f'\nLancement entraînement — {CFG.model_type} — {CFG.max_epochs} epochs')
print(f'Sortie : {CFG.out_dir}\n')

trainer = Trainer(CFG)
results = trainer.fit(loaders, resume=False)

print('\n══ Entraînement terminé ══')
print(f'Meilleurs résultats : {results}')

## 9 — Évaluation sur le jeu de test

In [ ]:
ckpt_path = CFG.out_dir / 'best.pth'
if not ckpt_path.exists():
    raise FileNotFoundError(f'Pas de checkpoint dans {ckpt_path} — lancer la cellule 8 d\'abord.')

gen_eval = build_generator(CFG).to(DEVICE)
ckpt     = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
gen_eval.load_state_dict(ckpt.get('gen_ema', ckpt['gen']))
print('[Éval] poids', 'EMA' if 'gen_ema' in ckpt else 'bruts', 'chargés.')
gen_eval.eval()

tr_eval = MetricsTracker(CFG.metric_data_range)
with torch.no_grad():
    for x, y, seg, meta in tqdm(loaders['test'], desc='Test', ncols=80):
        x = x.to(DEVICE)
        with autocast(device_type=AMP_DEVICE): pred = gen_eval(x)
        pn, yn = pred.cpu().float().numpy(), y.float().numpy()
        for b in range(pn.shape[0]):
            ht = bool(meta['has_tumour'][b]) if isinstance(meta['has_tumour'], torch.Tensor) else meta['has_tumour'][b]
            tr_eval.update(pn[b,0], yn[b,0], ht)

stats = tr_eval.summary()
print('\n── Résultats test ──────────────────────────')
for m in ('ssim', 'psnr', 'mae'):
    v = stats['all'][m]
    if v['n'] > 0:
        print(f'  {m:5s} = {v["mean"]:.4f} ± {v["std"]:.4f}  (n={v["n"]})')

## 10 — Visualisation

In [ ]:
# ── Courbes d'entraînement ────────────────────────────────────────────────────
log_csv = CFG.out_dir / 'train_log.csv'
if log_csv.exists():
    df = pd.read_csv(log_csv)
    fig, ax = plt.subplots(1, 3, figsize=(17, 4))
    ax[0].plot(df.epoch, df.loss_g, color='royalblue', label='L_G')
    ax[0].set(title='Generator Loss', xlabel='Epoch'); ax[0].legend()
    ax[1].plot(df.epoch, df.loss_d, color='tomato', label='L_D')
    ax[1].set(title='Discriminator Loss', xlabel='Epoch'); ax[1].legend()
    ax[2].plot(df.epoch, df.ssim, color='green', label='SSIM val')
    ax2b = ax[2].twinx()
    ax2b.plot(df.epoch, df.psnr, color='purple', ls='--', alpha=0.7, label='PSNR val')
    ax[2].set(title='Métriques de validation', xlabel='Epoch')
    ax[2].legend(loc='upper left'); ax2b.legend(loc='lower right')
    plt.suptitle(f'Entraînement — {CFG.out_name}', fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()

# ── Grille de prédiction ──────────────────────────────────────────────────────
test_info = splits.get('test', splits.get('val'))
test_ds   = MMITBraTS2DDataset(
    test_info['data_dir'], test_info['pids'],
    CFG.translation_pair, augment=False, cfg=CFG
)

if len(test_ds) == 0:
    print('Aucune tranche de test disponible.')
else:
    x_t, y_t, _, meta_t = test_ds[len(test_ds)//2]
    with torch.no_grad():
        with autocast(device_type=AMP_DEVICE):
            pred_t = gen_eval(x_t.unsqueeze(0).to(DEVICE))[0,0].cpu().float().numpy()

    src_name, tgt_name = PAIR_LABELS.get(CFG.translation_pair, ('Source','Cible'))
    gt_np, src_np = y_t[0].numpy(), x_t[0].numpy()
    diff = pred_t - gt_np; vm = max(abs(diff.min()), abs(diff.max()), 1e-8)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(src_np,  cmap='gray', vmin=-1, vmax=1); axes[0].set_title(f'Source: {src_name}');    axes[0].axis('off')
    axes[1].imshow(gt_np,   cmap='gray', vmin=-1, vmax=1); axes[1].set_title(f'Cible GT: {tgt_name}', color='limegreen'); axes[1].axis('off')
    axes[2].imshow(pred_t,  cmap='gray', vmin=-1, vmax=1); axes[2].set_title(f'Prédit: {tgt_name}', color='dodgerblue'); axes[2].axis('off')
    axes[3].imshow(diff, cmap='RdBu_r', vmin=-vm, vmax=vm); axes[3].set_title('Résidu', color='tomato'); axes[3].axis('off')

    m = gt_np!=0; dr=2.0
    ss = float(skimage_ssim(pred_t, gt_np, data_range=dr)) if m.any() else 0.
    ps = float(skimage_psnr(gt_np, pred_t, data_range=dr)) if m.any() else 0.
    plt.suptitle(f"Patient {meta_t['pid']} | z={meta_t['z']}   SSIM={ss:.4f}   PSNR={ps:.2f}dB",
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(CFG.out_dir/'prediction_grid.png', dpi=150, bbox_inches='tight')
    plt.show()